In [ ]:
# Imports en paden
import os
import subprocess
import time
import pandas as pd
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

script_locatie = os.getcwd()
doel_map = os.path.abspath(os.path.join(script_locatie, "..", "..", "data", "mobiliteitsBevragingHOGENT"))
temp_download_pad = os.path.join(script_locatie, "temp_downloads")
automation_data_path = os.path.join(script_locatie, "selenium_data")

os.makedirs(doel_map, exist_ok=True)
os.makedirs(temp_download_pad, exist_ok=True)
os.makedirs(automation_data_path, exist_ok=True)

In [9]:
# Forceer google om alle subprocessen af te sluiten
try:
    subprocess.run(["taskkill", "/F", "/IM", "chrome.exe", "/T"], capture_output=True)
    subprocess.run(["taskkill", "/F", "/IM", "chromedriver.exe", "/T"], capture_output=True)
except:
    pass

# Opties om vertraging tegen te gaan
chrome_options = Options()
chrome_options.add_argument(f"--user-data-dir={automation_data_path}")
chrome_options.add_argument("--profile-directory=Default")
chrome_options.add_argument("--remote-debugging-port=9222")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

prefs = {
    "download.default_directory": temp_download_pad,
    "download.prompt_for_download": False,
    "directory_upgrade": True
}
chrome_options.add_experimental_option("prefs", prefs)

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

url = "https://forms.office.com/Pages/DesignPageV2.aspx?subpage=design&token=47148982a16444afbdc55c901ef7933c&id=DjH3XBoJxUus1ybHIdTMzVjyleB1eNpMqLpd6hQbwI9UNjhPQlVKN0pJMzlCT1E2NVU2TVpaVEVYMS4u&analysis=true"
driver.get(url)

# Vind download in excel knop en klik erop
wacht = WebDriverWait(driver, 60)
excel_knop = wacht.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label='Een kopie in Excel downloaden']")))
excel_knop.click()

time.sleep(2)
driver.quit()

In [ ]:
# Bestand verwerken met pandas en opslaan in dataframe. Daarna naar data map
bestanden = [os.path.join(temp_download_pad, f) for f in os.listdir(temp_download_pad) if f.endswith(('.xlsx', '.csv'))]

if bestanden:
    nieuwste_bestand = max(bestanden, key=os.path.getmtime)
    
    if nieuwste_bestand.endswith('.xlsx'):
        df = pd.read_excel(nieuwste_bestand)
    else:
        df = pd.read_csv(nieuwste_bestand)

    if 'Begintijd' in df.columns:
        df['Begintijd'] = pd.to_datetime(df['Begintijd'])
        df['Datekey'] = df['Begintijd'].dt.strftime('%Y%m%d')

    bestandsnaam_output = "mobiliteitsbevraging_studenten_opgeschoond.csv"
    volledig_output_pad = os.path.join(doel_map, bestandsnaam_output)
    
    df.to_csv(volledig_output_pad, index=False)
else:
    print("Geen bestanden gevonden in de downloadmap.")

In [12]:
df.head()

,ID,Begintijd,Tijd van voltooien,E-mail,Naam,Tijd van laatste wijziging,Datum (format: YYYYMMDD),Vervoermiddel,Aantal km,Datekey
0,3,2026-02-17 09:19:27,2026-02-17 09:19:54,thomas.parmentier@hogent.be,Thomas Parmentier,NaN,20260217,Fiets,1,20260217
1,4,2026-02-17 09:19:57,2026-02-17 09:21:55,thomas.parmentier@hogent.be,Thomas Parmentier,NaN,20260217,Openbaar vervoer - trein,16,20260217
2,5,2026-02-17 09:21:57,2026-02-17 09:22:23,thomas.parmentier@hogent.be,Thomas Parmentier,NaN,20260217,Te voet,1,20260217
3,6,2026-02-17 10:31:09,2026-02-17 10:32:13,lennaert.deschepper@student.hogent.be,Lennaert De Schepper,NaN,20260213,Fiets,2,20260217
4,7,2026-02-17 10:32:24,2026-02-17 10:33:53,anqi.huang@student.hogent.be,Anqi Huang,NaN,20260217,Fiets,24,20260217


# Opmerkingen
- Selenium maakt heel veel temp bestanden aan deze worden in de gitignore gezet
- Het kan zijn dat je de eerste keer handmatig moet inloggen op je microsoft hogent account. Vanaf de tweede keer zal dit automatisch gebeuren